In [1]:
from Bio import Phylo, Entrez
from ete4 import PhyloTree
from ete4.treeview import TreeStyle, NodeStyle, AttrFace, TextFace, RectFace
from ete4 import NCBITaxa
import re
import time

In [2]:
def parse_assembly_id(s: str) -> str:
    m = re.search(r'\.\d+_', s)
    if m:
        return s[:m.end()-1]
    return

def find_dynamic_outgroup_taxids(ncbi_db, taxid_list):
    """
    Finds the most distantly related group of taxids based on NCBI lineages.
    """
    
    # Convert to integers for NCBI lookup
    taxids_int = [int(tid) for tid in taxid_list if tid is not None]
    lineages = ncbi_db.get_lineage_translator(taxids_int)
    
    valid_taxids = list(lineages.keys())
    if not valid_taxids:
        return []
        
    # Find the common lineage
    common_lineage = lineages[valid_taxids[0]]
    for t in valid_taxids[1:]:
        common_lineage = [val for val, val_cmp in zip(common_lineage, lineages[t]) if val == val_cmp]
        
    lca_taxid = common_lineage[-1]
    
    # Group the taxids by the first major evolutionary split after the LCA
    basal_splits = {}
    for t in valid_taxids:
        lin = lineages[t]
        lca_idx = lin.index(lca_taxid)
        
        # Get the node right after the LCA, or the LCA itself if it's the end of the line
        diverging_node = lin[lca_idx + 1] if lca_idx + 1 < len(lin) else lin[lca_idx]
        
        if diverging_node not in basal_splits:
            basal_splits[diverging_node] = []
        basal_splits[diverging_node].append(str(t)) # Convert back to string to match your keys
        
    # Assume the deepest branch with the fewest taxa is the outgroup 
    outgroup_taxa_branch = min(basal_splits.keys(), key=lambda k: len(basal_splits[k]))
    
    return basal_splits[outgroup_taxa_branch]


def assemblies_to_taxids(accessions, batch_size=100):
    taxid_map = {}

    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i+batch_size]
        query = " OR ".join([f"{acc}[Assembly Accession]" for acc in batch])

        handle = Entrez.esearch(
            db="assembly",
            term=query,
            retmax=len(batch)
        )
        search_record = Entrez.read(handle)
        handle.close()

        uids = search_record["IdList"]
        if not uids:
            continue

        handle = Entrez.esummary(
            db="assembly",
            id=",".join(uids),
            retmode="xml",
            report="full"

        )
        summary_record = Entrez.read(handle, validate=False)
        handle.close()

        docs = summary_record["DocumentSummarySet"]["DocumentSummary"]

        for doc in docs:
            accession = doc.get("AssemblyAccession")
            if accession.startswith("GCF_"):
                accession = accession.replace("GCF_", "GCA_")
            taxid = doc.get("Taxid")

            if accession and taxid:
                taxid_map[accession] = taxid

        time.sleep(2)

    return taxid_map

def set_layout(node):
    if node.is_leaf:      
        face = TextFace(node.name, fsize=10)
        face.margin_right = 12
        face.rotation = 179
        node.add_face(face, column=0, position="aligned")

In [4]:
ncbi = NCBITaxa()
Entrez.email = "p.vychik@gmail.com" # add email for sending large batches of requests
taxa = {}
assemblies_ids = {}
single_fetched = {}
# MetaInvert tree path
t = PhyloTree(open("./MetaInvert.treefile").read(), parser=1)

for node in t.traverse():
    if node.is_leaf:
        # genome assemble id require different conversion procedure
        if node.name.startswith("GCA_"):
            assembly_id = parse_assembly_id(node.name)
            if assembly_id:
                search_res = assemblies_to_taxids([assembly_id])
                if len(search_res):
                    taxid = list(search_res.values())[0]
                    single_fetched[node.name] = {"assembly_id": assembly_id,
                                                "taxid": taxid,
                                                "taxon_name": ncbi.get_taxid_translator([taxid])[int(taxid)]
                                                }
            continue
            
        node_name_cols = node.name.split("_")
        taxon_name = f"{node_name_cols[1]} {node_name_cols[2]}"
        tax_id_entry = ncbi.get_name_translator([taxon_name])
        taxa[node.name] = {"taxon_name": taxon_name,
                          "taxid": str(tax_id_entry[taxon_name][0]) if len(tax_id_entry) else None}

In [5]:
len(single_fetched) == len(set([entry["taxid"] for entry in single_fetched.values()]))

True

In [6]:
merged_taxa = {**taxa, **single_fetched}
len(merged_taxa) == len(set([entry["taxid"] for _, entry in merged_taxa.items()]))

False

In [7]:
p = "./merged_soil_tol_taxa_filter.phyloprofile"

# load taxa ids from gene trees
gene_trees_taxa_ids = set()
with open(p, "r") as fin:
    for line in fin.readlines():
        if "@" in line:
            tax_id = line.split("@")[1]
            gene_trees_taxa_ids.add(tax_id)

In [8]:
tree_ids = set()
t = PhyloTree(open("./MetaInvert.treefile").read(), parser=1)
for node in t.traverse():
    if node.is_leaf:
        tree_ids.add(merged_taxa[node.name]["taxid"])
        
keep_taxids = set.intersection(tree_ids, set(gene_trees_taxa_ids))
keep_node_names = [k for k,v in merged_taxa.items() if v["taxid"] in keep_taxids]
t.prune(keep_node_names)

# assign names to nodes
for node in t.traverse():
    if node.is_leaf:
        node.name = merged_taxa[node.name]["taxon_name"]

for node in t.traverse("postorder"):
    if not node.is_leaf:
        children = node.get_children()
        
        if len(children) == 2 and all(c.is_leaf for c in children):
            if children[0].name == children[1].name:
                # keep first
                children[1].delete()
                if len(node.get_children()) == 1:
                    node.delete(prevent_nondicotomic=False)

outgroup_taxids = find_dynamic_outgroup_taxids(ncbi, keep_taxids)
if outgroup_taxids:
    # map the outgroup taxids to their new leaf names in the tree
    outgroup_leaf_names = [merged_taxa[name]["taxon_name"] for name in keep_node_names 
                           if merged_taxa[name]["taxid"] in outgroup_taxids]
    
    # Grab the actual node objects from the ete4 tree
    outgroup_nodes = []
    for leaf_name in outgroup_leaf_names:
        found = t.search_nodes(name=leaf_name)
        if found:
            outgroup_nodes.extend(found)
            
    # Set the root
    if len(outgroup_nodes) == 1:
        t.set_outgroup(outgroup_nodes[0])
    elif len(outgroup_nodes) > 1:
        print(f"Rooting tree on clade containing {len(outgroup_nodes)} outgroup taxa.")
        lca_node = t.get_common_ancestor(outgroup_nodes)
        t.set_outgroup(lca_node)

t.write(outfile="pruned_metainvert_tree.nwk", parser=0) 

In [9]:
ts = TreeStyle()
ts.mode = "r"
ts.show_leaf_name = False
ts.layout_fn = set_layout
ts.margin_right = 200
ts.force_topology = False
ts.orientation = 1
ts.draw_guiding_lines = True
tree_svg_file = f"pruned_metainvert_tree_new_root.svg"
t.reverse_children()
t.render(tree_svg_file, w=600, units="px", tree_style=ts)

Wrote file: pruned_metainvert_tree_new_root.svg


In [10]:
# save pruned version with ncbi id for PhyloproProfile taxa ordering
species_to_taxid = { entry["taxon_name"]:entry["taxid"] for _, entry in merged_taxa.items()}
for node in t.traverse():
    if node.is_leaf:
        if node.name in species_to_taxid:
            node.name = f"ncbi{species_to_taxid[node.name]}"
        else:
            print(node.name)
t.write(outfile="pruned_tree_phyloprofile_sorting_root.nwk", parser=9)